In [ ]:
import matplotlib
# matplotlib.use('tkAgg')
%matplotlib inline
from matplotlib import pyplot as plt
from PIL import Image
import torch
from transformers import GLPNImageProcessor, GLPNForDepthEstimation
import numpy as np
import open3d as o3d

# Materials used:
- https://amanjaglan.medium.com/generating-3d-images-and-point-clouds-with-python-open3d-and-glpn-for-depth-estimation-a0a484d77570
- https://towardsdatascience.com/generate-a-3d-mesh-from-an-image-with-python-12210c73e5cc/
- https://www.open3d.org/docs/latest/tutorial/geometry/surface_reconstruction.html

- GLPN model - https://huggingface.co/vinvino02/glpn-nyu

In [ ]:
image_path = '/home/maciej/dev/depth/422b1df6e7202cdef0cf8824c206138f.jpg'

In [ ]:

def glpn_processor(image_path, processor):
    image = Image.open(image_path)
    print(f"image size: {image.size}") # PIL gives (width, height)

    # feature_extractor dopasowuje rozmiar zdjęcia, normalizuje piksele (0-255 -> 0-1 i standaryzacja)
    # oraz zamienia format HWC (Height, Width, Channels) na format PyTorcha: CHW.
    # 'return_tensors="pt"' zwraca PyTorch Tensors i dodaje wymiar Batch Size: [1, C, H, W].
    inputs = processor(images=image, return_tensors='pt')

    return image, inputs

def glpn_depth_prediction(inputs, model):

    # torch.no_grad() blokuje zapamiętywanie operacji w grafie obliczeniowym.
    # stops calculating gradients, which reduces memory usage since we dont need to train model, we only want to do inference.
    with torch.no_grad():

        outputs = model(**inputs)

        predicted_depth = outputs.predicted_depth
        
    print(f"predicted_depth shape: {predicted_depth.shape}")

    return predicted_depth

processor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu")

image, inputs = glpn_processor(image_path, processor)
predicted_depth = glpn_depth_prediction(inputs, model)

# depth_anything model

In [ ]:
# Example with Depth Anything V2
from transformers import pipeline

image_anth = Image.open(image_path)

pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Large-hf")
depth = pipe(image_anth)

predicted_depth_anth = depth['predicted_depth']

In [ ]:
# from transformers import AutoImageProcessor, AutoModelForDepthEstimation
# import torch
# import numpy as np
# from PIL import Image
# import httpx
# from io import BytesIO

# image_processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
# model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf", device_map="auto")

# image = Image.open(image_path)

# # prepare image for the model
# inputs = image_processor(images=image, return_tensors="pt")

# with torch.no_grad():
#     outputs = model(**inputs)

# # interpolate to original size
# post_processed_output = image_processor.post_process_depth_estimation(
#     outputs,
#     target_sizes=[(image.height, image.width)],
# )

# # visualize the prediction
# predicted_depth = post_processed_output[0]["predicted_depth"]
# depth = predicted_depth * 255 / predicted_depth.max()
# depth = depth.detach().cpu().numpy()
# depth = Image.fromarray(depth.astype("uint8"))

In [ ]:
def pad_edge(predicted_depth, image, pad=0):
    predicted_depth = predicted_depth.squeeze().cpu().numpy() * 1000.0

    predicted_depth = predicted_depth[pad:-pad, pad:-pad]

    depth_h, depth_w = predicted_depth.shape
    
    left = (image.width - depth_w) // 2
    top = (image.height - depth_h) // 2
    image = image.crop((left, top, left + depth_w, top + depth_h))

    print(f"depth shape: {predicted_depth.shape}")   # (H, W)
    print(f"image size:  {image.size}")              # (W, H) in PIL → should match

    return predicted_depth, image

predicted_depth, image = pad_edge(predicted_depth, image, pad = 0)

print(predicted_depth.shape)
print(image.size)

In [ ]:
def compare_depth_map(predicted_depth, image, title="depth model"):
    fig, ax = plt.subplots(1, 2, figsize=(10, 6))
    fig.suptitle(title)
    ax[0].imshow(image)
    ax[0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax[1].imshow(predicted_depth, cmap='plasma')
    ax[1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    plt.tight_layout()


compare_depth_map(predicted_depth, image, "GLPN model prediction")
compare_depth_map(predicted_depth_anth, image_anth, "DEPTH_ANYTHING model prediction")

In [ ]:
def color_array_from_image(image):

    color_array = np.array(image)

    print(f"color_array.shape: {color_array.shape}")
    return color_array

In [ ]:
def create_point_cloud_from_depth_image(depth_image, intrinsic, color_array, scale=1.0):

    # handle torch tensor input
    if hasattr(depth_image, 'numpy'):
        depth_image = depth_image.squeeze().numpy()
    
    height, width = depth_image.shape
    print(height, width)
    depth_image = o3d.geometry.Image((depth_image / scale).astype(np.float32))    # Create an RGBD image
    print(f'depth_image: {depth_image}')

    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
        # color=o3d.geometry.Image(np.zeros((height, width, 3), dtype=np.uint8)),
        color = o3d.geometry.Image(color_array),
        depth=depth_image,
        convert_rgb_to_intensity=False
    )    
    print(f'rgbd_image: {rgbd_image}')

# Create the point cloud from the RGBD image
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_image,
        intrinsic
    )    

    print(f'pcd: {pcd}')
    return pcd

In [ ]:
fx = 1000  # Focal length in the x axis
fy = 1000  # Focal length in the y axis
cx = predicted_depth.shape[1] / 2  # Principal point in the x axis
cy = predicted_depth.shape[0] / 2  # Principal point in the y axis
intrinsic = o3d.camera.PinholeCameraIntrinsic(
    width=predicted_depth.shape[1],
    height=predicted_depth.shape[0],
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy
)

In [ ]:
color_array = color_array_from_image(image)
# Create the point cloud from the depth image
pcd = create_point_cloud_from_depth_image(predicted_depth, intrinsic, color_array=color_array, scale=1.0)
# Visualize the point cloud
o3d.visualization.draw_plotly([pcd])

In [ ]:
# predicted_depth_anth, image_anth = pad_edge(predicted_depth_anth, image_anth, pad = 16)

color_array = color_array_from_image(image_anth)

pcd_anth = create_point_cloud_from_depth_image(predicted_depth_anth, intrinsic, color_array=color_array, scale=1.0)

o3d.visualization.draw_plotly([pcd_anth])

In [ ]:
print(np.asarray(pcd.colors).shape)

In [ ]:
def create_aplha_shape(pcd,alpha=0.05):
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
    )
    pcd.orient_normals_towards_camera_location(camera_location=[0, 0, 1])
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
    mesh.compute_vertex_normals()

        # correct way to transfer colors
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    vertex_colors = []
    for vertex in np.asarray(mesh.vertices):
        _, idx, _ = pcd_tree.search_knn_vector_3d(vertex, 1)  # find nearest point
        vertex_colors.append(np.asarray(pcd.colors)[idx[0]])
    
    mesh.vertex_colors = o3d.utility.Vector3dVector(np.array(vertex_colors))

    # o3d.visualization.draw_plotly([mesh])

    o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

create_aplha_shape(pcd=pcd)


In [ ]:
def create_ball_pivoting(pcd):

    distances = pcd.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    print(f"avg point spacing: {avg_dist:.5f}")

    radii = [avg_dist * 1.5, avg_dist * 3, avg_dist * 6, avg_dist * 12]

    rec_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd, o3d.utility.DoubleVector(radii)
    )

    rec_mesh.compute_vertex_normals()
    o3d.visualization.draw_plotly([rec_mesh])

create_ball_pivoting(pcd)

In [ ]:
def create_poisson_reconstruction(pcd):
    # depth=9 or 10 gives more detail, lower = smoother
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)

    # Poisson always outputs a closed mesh — it invents geometry to fill holes.
    # The problem: invented geometry is often garbage. Trim it by density.
    densities = np.asarray(densities)
    threshold = np.percentile(densities, 10)  # remove lowest 10% density vertices
    vertices_to_remove = densities < threshold
    mesh.remove_vertices_by_mask(vertices_to_remove)

    mesh.compute_vertex_normals()
    o3d.visualization.draw_plotly([mesh])

create_poisson_reconstruction(pcd)

In [ ]:
mesh = o3d.io.read_triangle_mesh("temp_mesh.ply")

print("vertices:", np.asarray(mesh.vertices).shape)
print("vertex colors:", np.asarray(mesh.vertex_colors).shape)  # is it (N, 3) or (0, 3)?
print("triangles:", np.asarray(mesh.triangles).shape)

# check color range
if mesh.has_vertex_colors():
    colors = np.asarray(mesh.vertex_colors)
    print("color min:", colors.min())
    print("color max:", colors.max())

In [ ]:
import open3d as o3d
pcd_check = o3d.io.read_point_cloud("temp_pcd.ply")
print("pcd colors:", np.asarray(pcd_check.colors).shape)  # should NOT be (0, 3)

In [ ]:
import pymeshlab

def create_poisson_reconstruction_pymeshlab(pcd):
    # fix normals in open3d first — more reliable
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
    )
    pcd.orient_normals_towards_camera_location(camera_location=[0, 0, 1])

    # save with normals already computed
    o3d.io.write_point_cloud("temp_pcd.ply", pcd)

    ms = pymeshlab.MeshSet()
    ms.load_new_mesh("temp_pcd.ply")

    # skip compute_normal_for_point_clouds — normals already in the file
    ms.generate_surface_reconstruction_screened_poisson(
        depth=9,
        scale=1.1,
        samplespernode=1.5,
    )

    # trim low density garbage
    ms.meshing_remove_connected_component_by_face_number(mincomponentsize=500)

    ms.save_current_mesh("temp_mesh.ply")

    mesh = o3d.io.read_triangle_mesh("temp_mesh.ply")
    mesh.compute_vertex_normals()

    # sanity checks
    print("vertices:", np.asarray(mesh.vertices).shape)
    print("has colors:", mesh.has_vertex_colors())

    mesh = o3d.io.read_triangle_mesh("temp_mesh.ply")
    mesh.compute_vertex_normals()

    # try this instead of draw_plotly
    o3d.visualization.draw_geometries([mesh])

create_poisson_reconstruction_pymeshlab(pcd)